# 09 — Explanation Evaluation

Explanations can be wrong in subtle ways. Rigorous evaluation is necessary before deploying XAI in production.

## Faithfulness, Robustness, and Stability

**Faithfulness** (fidelity): does the explanation reflect what the model actually computes?
- *Sufficiency*: if you set features to their attributed values, does the prediction change accordingly?
- *Necessity*: if you remove the top-attributed features, does the prediction drop?
- *Pixel/feature flipping*: progressively remove the most important features; a faithful explainer should cause the prediction to drop fastest

**Robustness**: similar inputs should produce similar explanations
- *Input stability*: perturb the input slightly; the explanation should barely change
- *Adversarial manipulability*: it's possible to craft inputs where the prediction is unchanged but the explanation completely changes (Slack et al., 2020 — adversarial LIME/SHAP)

**Stability** (consistency): the same input should produce the same explanation on repeated calls
- Deterministic methods (SHAP TreeExplainer, IG) have perfect stability
- Stochastic methods (LIME, KernelSHAP) have variance that must be quantified

**Alignment with domain knowledge**: for structured domains (e.g., credit risk, medical diagnosis), top features should be known causal drivers — a sanity check against expert knowledge.

In [ ]:
# Explanation quality benchmark
import numpy as np
import torch
import torch.nn as nn
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
torch.manual_seed(42)

# Dataset where ground-truth important features are known
n, n_features = 1000, 10
# True signal only in features 0-3; 4-9 are noise
X, _ = make_classification(n_samples=n, n_features=n_features,
                            n_informative=4, n_redundant=0, n_repeated=0,
                            n_clusters_per_class=1, random_state=42)
y_true = (X[:, 0] + X[:, 1] - X[:, 2] > 0).astype(int)

scaler = StandardScaler()
X = scaler.fit_transform(X)
feature_names = [f'f{i}' for i in range(n_features)]
# Ground truth: features 0, 1, 2 matter most
true_important = {0, 1, 2}

Xtr = torch.tensor(X[:800], dtype=torch.float32)
ytr = torch.tensor(y_true[:800], dtype=torch.float32)
Xte = torch.tensor(X[800:], dtype=torch.float32)

class TwoLayerNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(10, 32), nn.ReLU(),
            nn.Linear(32, 1), nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

model = TwoLayerNet()
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
for _ in range(100):
    loss = nn.BCELoss()(model(Xtr), ytr)
    opt.zero_grad(); loss.backward(); opt.step()

model.eval()
with torch.no_grad():
    acc = ((model(Xte) > 0.5).float() == torch.tensor(y_true[800:], dtype=torch.float32)).float().mean()
print(f'Model accuracy: {acc:.3f}')

In [ ]:
# Faithfulness metric: feature flipping
def feature_flip_curve(model, x, attribution, n_steps=10, mode='remove'):
    """
    Progressively remove top-attributed features and measure prediction drop.
    mode='remove': replace with baseline (zero); mode='add': start from baseline
    """
    sorted_idx = np.argsort(np.abs(attribution))[::-1]
    scores = []
    x_cur = x.clone()

    with torch.no_grad():
        scores.append(model(x_cur.unsqueeze(0)).item())
        for k in range(1, n_steps + 1):
            idx = sorted_idx[k - 1]
            if mode == 'remove':
                x_cur = x_cur.clone()
                x_cur[idx] = 0.0  # replace with baseline
            scores.append(model(x_cur.unsqueeze(0)).item())
    return scores

# Compare random attribution vs IG
def integrated_gradients_simple(model, x, n_steps=50):
    baseline = torch.zeros_like(x)
    alphas = torch.linspace(0, 1, n_steps + 1)
    interp = baseline + alphas.unsqueeze(1) * (x - baseline)
    interp.requires_grad_(True)
    preds = model(interp)
    grads = []
    for i in range(len(alphas)):
        if interp.grad is not None: interp.grad.zero_()
        preds[i].backward(retain_graph=True)
        grads.append(interp.grad[i].clone())
    grads = torch.stack(grads)
    return ((x - baseline) * grads[:-1].mean(0)).detach().numpy()

x_sample = Xte[0]
ig_attrs = integrated_gradients_simple(model, x_sample.clone())
rand_attrs = np.random.randn(10)

ig_curve = feature_flip_curve(model, x_sample, ig_attrs)
rand_curve = feature_flip_curve(model, x_sample, rand_attrs)

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(ig_curve, 'o-', label='IG attribution', color='steelblue')
ax.plot(rand_curve, 's--', label='Random attribution', color='tomato')
ax.set_xlabel('Features removed (most important first)')
ax.set_ylabel('Model prediction')
ax.set_title('Faithfulness: Feature Flipping Curve')
ax.legend()
plt.tight_layout()
plt.savefig('/tmp/faithfulness.png', dpi=100, bbox_inches='tight')
plt.show()
print('Faithfulness: IG should drop faster than random')

In [ ]:
# Stability metric: AOPC (area over perturbation curve)
def aopc(curve):
    """Area over perturbation curve — higher means more faithful."""
    baseline = curve[0]
    return sum(baseline - c for c in curve[1:]) / len(curve[1:])

ig_aopc = aopc(ig_curve)
rand_aopc = aopc(rand_curve)
print(f'AOPC (IG): {ig_aopc:.4f}')
print(f'AOPC (Random): {rand_aopc:.4f}')
print(f'IG is {ig_aopc/max(rand_aopc,1e-6):.1f}x more faithful than random')

# Ground-truth alignment: which features does IG rank as most important?
top_k = set(np.argsort(np.abs(ig_attrs))[::-1][:3])
precision_at_3 = len(top_k & true_important) / 3
print(f'\nIG top-3 features: {[f"f{i}" for i in top_k]}')
print(f'True important: {[f"f{i}" for i in true_important]}')
print(f'Precision@3 (ground truth alignment): {precision_at_3:.2f}')